## **Preprocess Video to Silhouettes**

In [ ]:
import cv2
import os
import shutil
import numpy as np
from ultralytics import YOLO

def process_video_to_silhouettes_custom(
    video_path="temp_gait.avi", 
    output_folder="live_test_folder", 
    img_size=(72, 72),
    
    inference_size=1080,        # Increase to 1280 or 1920 for maximum sharpness (runs slower)
    confidence_threshold=0.8,   # Increase to 0.6 or 0.7 for stricter, cleaner edges
    extract_every_n_frame=3     # 1 = 30 FPS, 2 = 15 FPS, 3 = 10 FPS, 6 = 5 FPS
):
    if os.path.exists(output_folder):
        shutil.rmtree(output_folder)
    os.makedirs(output_folder)
    
    print("Loading YOLOv8 Segmentation Model...")
    model = YOLO("yolov8n-seg.pt")
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return
        
    frame_count = 0
    saved_count = 0
    print(f"Processing video with Inference Size: {inference_size}, saving every {extract_every_n_frame} frames...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            break
            
        frame_count += 1
        
        # Frame Skipping Logic: If it's not the Nth frame, skip the AI processing entirely
        if frame_count % extract_every_n_frame != 0:
            continue
        
        # We pass your custom inference_size (imgsz) and confidence (conf) here
        results = model.predict(
            frame, 
            classes=[0], 
            retina_masks=True, 
            imgsz=inference_size, 
            conf=confidence_threshold,
            verbose=False
        )
        
        binary_silhouette = np.zeros((frame.shape[0], frame.shape[1]), dtype=np.uint8)
        
        for r in results:
            if r.masks is not None:
                mask_data = r.masks.data[0].cpu().numpy()
                binary_silhouette = np.maximum(binary_silhouette, (mask_data > 0.5).astype(np.uint8) * 255)

        if np.sum(binary_silhouette) > 0:
            x, y, w, h = cv2.boundingRect(binary_silhouette)
            
            pad_w = int(w * 0.10)
            pad_h = int(h * 0.10)
            
            x1 = max(x - pad_w, 0)
            y1 = max(y - pad_h, 0)
            x2 = min(x + w + pad_w, frame.shape[1])
            y2 = min(y + h + pad_h, frame.shape[0])
            
            cropped_person = binary_silhouette[y1:y2, x1:x2]
            final_silhouette = cv2.resize(cropped_person, img_size)
            
            save_path = os.path.join(output_folder, f"frame_{saved_count:04d}.png")
            cv2.imwrite(save_path, final_silhouette)
            saved_count += 1

    cap.release()
    print(f"\n[SUCCESS] Extracted {saved_count} perfectly spaced silhouettes to '{output_folder}'!")

if __name__ == "__main__":
    current_dir = os.getcwd()
    absolute_video_path = os.path.join(current_dir, "Screen Recording 2025-11-18 112016.mp4") 
    absolute_output_folder = os.path.join(current_dir, "live_test_folder")
    
    process_video_to_silhouettes_custom(
        video_path=absolute_video_path, 
        output_folder=absolute_output_folder, 
        img_size=(72, 72),
        inference_size=1080,       # Tweak me! Try 1920
        extract_every_n_frame=2    # Tweak me! Try 2 or 4
    )

Loading YOLOv8 Segmentation Model...
Processing video with Inference Size: 1080, saving every 2 frames...
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [1088]
WARNING imgsz=[1080] must be multiple of max stride 32, updating to [

## **Capture The Subject**

In [ ]:
import cv2
import time
import threading
from ultralytics import YOLO

def auto_capture_always_on_threaded_720p():
    # 1. Force 720p Resolution (The Sweet Spot for Speed vs. Quality)
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # THE HARDWARE AUTO-BENCHMARK
    print(f"\n[SYSTEM] Benchmarking camera hardware at {width}x{height}... Please wait 2 seconds.")
    start_time = time.time()
    for _ in range(20): 
        cap.read()
    elapsed_time = time.time() - start_time
    
    true_fps = 20 / elapsed_time
    print(f"[SYSTEM] Hardware benchmark complete. Actual Camera Speed: {true_fps:.1f} FPS")
    
    # Exactly 3 seconds of video
# Exactly 4 seconds of video to capture the full stride
    frames_to_capture = int(true_fps * 5.0)    
    # Dynamic skip targeting ~10-15 FPS for the final model
    dynamic_skip = max(1, round(true_fps / 10.0))
    
    print(f"[SYSTEM] -> Setting capture limit to {frames_to_capture} frames.")
    print(f"[SYSTEM] -> Setting extraction skip rate to {dynamic_skip}.")
    # ==========================================
    
    # 2. Tripwire Dimensions (60% width, 100% height)
    zone_w = int(width * 0.60)
    zone_h = int(height * 1.0)
    zx1 = int((width - zone_w) / 2)
    zy1 = 0; zx2 = zx1 + zone_w; zy2 = height
    
    print("\n[SYSTEM] Loading fast Watcher model...")
    watcher_model = YOLO("yolov8n.pt")
    
    # 3. System States
    state = "WAITING"
    frames_captured = 0
    subject_counter = 1
    cooldown_frames = 0
    
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = None
    filename = ""

    print("\n[READY] Always-On System Live. Waiting for subject...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        display_frame = frame.copy()

        if state == "WAITING":
            if cooldown_frames > 0:
                cooldown_frames -= 1
                cv2.putText(display_frame, "PROCESSING PREVIOUS...", (zx1 + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            else:
                results = watcher_model.predict(frame, classes=[0], verbose=False)
                person_in_zone = False

                for r in results:
                    for box in r.boxes:
                        px1, py1, px2, py2 = map(int, box.xyxy[0])
                        if not (px2 < zx1 or px1 > zx2 or py2 < zy1 or py1 > zy2):
                            person_in_zone = True
                            break

                cv2.rectangle(display_frame, (zx1, zy1), (zx2, zy2), (255, 0, 0), 2)
                cv2.putText(display_frame, "WAITING FOR SUBJECT...", (zx1 + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

                if person_in_zone:
                    print(f"\n[TRIGGER] Subject detected! Capturing for 3 seconds...")
                    state = "CAPTURING"
                    frames_captured = 0
                    filename = f"live_subject_{subject_counter}.avi"
                    out = cv2.VideoWriter(filename, fourcc, true_fps, (width, height))

        elif state == "CAPTURING":
            out.write(frame)
            frames_captured += 1
            
            cv2.rectangle(display_frame, (zx1, zy1), (zx2, zy2), (0, 0, 255), 4)
            cv2.putText(display_frame, f"CAPTURING: {frames_captured}/{frames_to_capture}", (zx1 + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

            if frames_captured >= frames_to_capture:
                print(f"[SUCCESS] Saved {filename}")
                out.release()
                
                folder_name = f"processed_subject_{subject_counter}"
                
                # The Thread gets the optimized 720p inference size
                worker = threading.Thread(
                    target=process_video_to_silhouettes_custom, 
                    kwargs={
                        "video_path": filename,
                        "output_folder": folder_name,
                        "img_size": (72, 72),
                        "inference_size": 720,  # <--- Matched to new camera resolution
                        "confidence_threshold": 0.7,
                        "extract_every_n_frame": dynamic_skip 
                    }
                )
                worker.start()
                
                subject_counter += 1
                state = "WAITING"
                cooldown_frames = int(true_fps) 

        cv2.imshow("Always-On Biometric Scanner", display_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    if out is not None: out.release()
    cv2.destroyAllWindows()
if __name__ == "__main__":
    auto_capture_always_on_threaded_720p()




[SYSTEM] Benchmarking camera hardware at 1280x720... Please wait 2 seconds.
[SYSTEM] Hardware benchmark complete. Actual Camera Speed: 9.5 FPS
[SYSTEM] -> Setting capture limit to 47 frames.
[SYSTEM] -> Setting extraction skip rate to 1.

[SYSTEM] Loading fast Watcher model...

[READY] Always-On System Live. Waiting for subject...

[TRIGGER] Subject detected! Capturing for 3 seconds...
[SUCCESS] Saved live_subject_1.avi


NameError: name 'process_video_to_silhouettes_custom' is not defined

## **Capturing The Subject And Testing At The Endpoint In Real Time**

In [ ]:
import cv2
import time
import threading
import os
import shutil
import requests
from ultralytics import YOLO

# GLOBAL VARIABLE: Shares the API result between threads
latest_prediction = {"text": "", "timestamp": 0}

# 1. THE NETWORK & CLEANUP FUNCTION
def send_to_endpoint_and_cleanup(folder_path, video_path, endpoint_url="http://140.245.221.62:8000/predict"):
    
    global latest_prediction
    print(f"\n[NETWORK] Packaging '{folder_path}' for transmission...")
    
    zip_filename = f"{folder_path}.zip"
    shutil.make_archive(folder_path, 'zip', folder_path)
    
    try:
        print(f"[NETWORK] POSTing {zip_filename} to endpoint...")
        with open(zip_filename, 'rb') as f:
            # UPDATED: Matches your FastAPI cURL perfectly!
            files = {'file': (zip_filename, f, 'application/x-zip-compressed')}
            response = requests.post(endpoint_url, files=files, timeout=10)
            
        if response.status_code == 200:
            data = response.json()
            print(f"[NETWORK SUCCESS] Endpoint Responded: {data}")
            
            # Extract the prediction and update the global variable for the camera UI!
            pred = data.get("prediction", "UNKNOWN")
            conf = data.get("confidence", 0.0)
            
            # Save it so the camera loop can draw it on the screen
            latest_prediction["text"] = f"MATCH: {pred} (Conf: {conf:.2f})"
            latest_prediction["timestamp"] = time.time()
            
        else:
            print(f"[NETWORK ERROR] Endpoint returned status code: {response.status_code}")
            
    except requests.exceptions.RequestException as e:
        print(f"[NETWORK FAILED] Could not reach endpoint. Error: {e}")

    print(f"[CLEANUP] Wiping temporary files for {folder_path}...")
    try:
        if os.path.exists(zip_filename): os.remove(zip_filename)
        if os.path.exists(video_path): os.remove(video_path)
        if os.path.exists(folder_path): shutil.rmtree(folder_path)
        print("[CLEANUP] System storage cleared.")
    except Exception as e:
        print(f"[CLEANUP ERROR] Failed to delete files: {e}")

# 2. THE THREAD MANAGER (The Wrapper)
def thread_pipeline_manager(video_filename, folder_name, skip_rate, api_url):
    # Step A: Run your exact YOLO extraction
    process_video_to_silhouettes_custom(
        video_path=video_filename,
        output_folder=folder_name,
        img_size=(72, 72),
        inference_size=720,
        confidence_threshold=0.7, 
        extract_every_n_frame=skip_rate
    )
    
    # Step B: Zip, Send, and Clean up!
    send_to_endpoint_and_cleanup(folder_name, video_filename, endpoint_url=api_url)

# 3. THE MAIN CAMERA ENGINE
def auto_capture_always_on_threaded_720p():
    global latest_prediction
    
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"\n[SYSTEM] Benchmarking camera hardware at {width}x{height}... Please wait 2 seconds.")
    start_time = time.time()
    for _ in range(20): cap.read()
    elapsed_time = time.time() - start_time
    
    true_fps = 20 / elapsed_time
    print(f"[SYSTEM] Hardware benchmark complete. Actual Camera Speed: {true_fps:.1f} FPS")
    
    frames_to_capture = int(true_fps * 5.0)    
    dynamic_skip = max(1, round(true_fps / 10.0))
    
    print(f"[SYSTEM] -> Setting capture limit to {frames_to_capture} frames.")
    print(f"[SYSTEM] -> Setting extraction skip rate to {dynamic_skip}.")
    
    zone_w = int(width * 0.60)
    zone_h = int(height * 1.0)
    zx1 = int((width - zone_w) / 2)
    zy1 = 0; zx2 = zx1 + zone_w; zy2 = height
    
    print("\n[SYSTEM] Loading fast Watcher model...")
    watcher_model = YOLO("yolov8n.pt")
    
    state = "WAITING"
    frames_captured = 0
    subject_counter = 1
    cooldown_frames = 0
    
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = None
    filename = ""

    print("\n[READY] Always-On System Live. Waiting for subject...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        display_frame = frame.copy()

        # UI OVERLAY: DISPLAY API RESULT ON SCREEN
        # If we received a prediction in the last 10 seconds, show it!
        if time.time() - latest_prediction["timestamp"] < 10.0:
            # Draw a black background box so the text is easy to read
            cv2.rectangle(display_frame, (30, 30), (700, 110), (0, 0, 0), -1)
            # Draw the bright green Prediction Text
            cv2.putText(display_frame, latest_prediction["text"], (50, 85), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        if state == "WAITING":
            if cooldown_frames > 0:
                cooldown_frames -= 1
                cv2.putText(display_frame, "PROCESSING PREVIOUS...", (zx1 + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            else:
                results = watcher_model.predict(frame, classes=[0], verbose=False)
                person_in_zone = False

                for r in results:
                    for box in r.boxes:
                        px1, py1, px2, py2 = map(int, box.xyxy[0])
                        if not (px2 < zx1 or px1 > zx2 or py2 < zy1 or py1 > zy2):
                            person_in_zone = True
                            break

                cv2.rectangle(display_frame, (zx1, zy1), (zx2, zy2), (255, 0, 0), 2)
                cv2.putText(display_frame, "WAITING FOR SUBJECT...", (zx1 + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

                if person_in_zone:
                    print(f"\n[TRIGGER] Subject detected! Capturing for 5 seconds...")
                    state = "CAPTURING"
                    frames_captured = 0
                    filename = f"live_subject_{subject_counter}.avi"
                    out = cv2.VideoWriter(filename, fourcc, true_fps, (width, height))

        elif state == "CAPTURING":
            out.write(frame)
            frames_captured += 1
            
            cv2.rectangle(display_frame, (zx1, zy1), (zx2, zy2), (0, 0, 255), 4)
            cv2.putText(display_frame, f"CAPTURING: {frames_captured}/{frames_to_capture}", (zx1 + 10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

            if frames_captured >= frames_to_capture:
                print(f"[SUCCESS] Saved {filename}")
                out.release()
                
                folder_name = f"processed_subject_{subject_counter}"
                my_api_url = "http://140.245.221.62:8000/predict" 
                
                worker = threading.Thread(
                    target=thread_pipeline_manager, 
                    args=(filename, folder_name, dynamic_skip, my_api_url)
                )
                worker.start()
                
                subject_counter += 1
                state = "WAITING"
                cooldown_frames = int(true_fps) 

        cv2.imshow("Always-On Biometric Scanner", display_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    if out is not None: out.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    auto_capture_always_on_threaded_720p()


[SYSTEM] Benchmarking camera hardware at 1280x720... Please wait 2 seconds.
[SYSTEM] Hardware benchmark complete. Actual Camera Speed: 9.5 FPS
[SYSTEM] -> Setting capture limit to 47 frames.
[SYSTEM] -> Setting extraction skip rate to 1.

[SYSTEM] Loading fast Watcher model...

[READY] Always-On System Live. Waiting for subject...

[TRIGGER] Subject detected! Capturing for 5 seconds...
[SUCCESS] Saved live_subject_1.avi
Loading YOLOv8 Segmentation Model...
Processing video with Inference Size: 720, saving every 1 frames...
WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
WARNING imgsz=[720] must be multiple of max stride 32, updating to [736]
WARNING imgsz=[720] must be mult

KeyboardInterrupt: 